<a href="https://colab.research.google.com/github/shin584/project/blob/ensemble_system/scanner%2Bensemble%2BUI_v6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 모델 다운로드&로드


In [1]:
!wget -O model.keras "https://github.com/shin584/project/raw/ensemble_system/models/Multi-Cas_1IN_9Hydra_Divide_Testset.keras"
!wget -O model_cas12a.keras "https://github.com/shin584/project/raw/ensemble_system/models/Cas12a_Only.keras"
# 2. PyTorch 뼈대 파일 다운로드 (1D_models 브랜치)
!wget -O adapters.py "https://raw.githubusercontent.com/shin584/project/1D_models/models/SaCas9_model/adapters.py"
!wget -O backbones.py "https://raw.githubusercontent.com/shin584/project/1D_models/models/SaCas9_model/backbones.py"
# 3. PyTorch SaCas9 5-Fold 가중치 다운로드 (1D_models 브랜치)
!wget -O best_model_fold1.pth "https://github.com/shin584/project/raw/1D_models/models/SaCas9_model/data/best_model_fold1.pth"
!wget -O best_model_fold2.pth "https://github.com/shin584/project/raw/1D_models/models/SaCas9_model/data/best_model_fold2.pth"
!wget -O best_model_fold3.pth "https://github.com/shin584/project/raw/1D_models/models/SaCas9_model/data/best_model_fold3.pth"
!wget -O best_model_fold4.pth "https://github.com/shin584/project/raw/1D_models/models/SaCas9_model/data/best_model_fold4.pth"
!wget -O best_model_fold5.pth "https://github.com/shin584/project/raw/1D_models/models/SaCas9_model/data/best_model_fold5.pth"


--2026-05-29 08:24:54--  https://github.com/shin584/project/raw/ensemble_system/models/Multi-Cas_1IN_9Hydra_Divide_Testset.keras
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/shin584/project/ensemble_system/models/Multi-Cas_1IN_9Hydra_Divide_Testset.keras [following]
--2026-05-29 08:24:54--  https://raw.githubusercontent.com/shin584/project/ensemble_system/models/Multi-Cas_1IN_9Hydra_Divide_Testset.keras
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1029645 (1006K) [application/octet-stream]
Saving to: ‘model.keras’

model.keras         100%[===================>]   1006K  --.-KB/s    in 0.05

In [2]:
import tensorflow as tf

model = tf.keras.models.load_model("model.keras")
model_cas12a = tf.keras.models.load_model("model_cas12a.keras")

# ngrok 패키지 설치

In [3]:
!pip install streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 73.3 MB/s eta 0:00:00


# pam_scanner.py 파일 저장

In [8]:
%%writefile /content/pam_scanner.py
"""
pam_scanner.py
──────────────
PAM 서열 탐색 전담 모듈.
"""

import re
import math
import unicodedata
from dataclasses import dataclass, field
from typing import Optional


# ════════════════════════════════════════════════
# 섹션 1. 설정
# ════════════════════════════════════════════════

@dataclass(frozen=True)
class CasConfig:
    name:               str
    pam_iupac:          str
    guide_len:          int
    pam_position:       str   # '3prime' | '5prime'
    cut_offset:         int
    model_path:         str
    model_input_len:    int

# [수정] SpCas9 PAM을 NGN(3bp) → NG(2bp)로 변경
# 이유: SpCas9-NG 변종의 PAM이 2bp 'NG'이므로, 3bp NGN으로 탐색하면
#       SpCas9-NG 전용 사이트가 누락될 수 있음.
#       NG는 NGN의 부분집합이 아니라 독립적인 2bp PAM이므로 별도 처리 필요.
#       단, 나머지 8개 변종(WT, VRQR 등)은 NGG/NGA 등 NGN에 해당하므로
#       NG(2bp)로 탐색해도 모두 포함됨 (NGG → NG + G, 탐색시 NG 패턴 매칭 성공).
CAS_CONFIGS: list[CasConfig] = [
    CasConfig(
        name            = "SpCas9",
        pam_iupac       = "NG",       # 변경: NGN → NG (2bp, 모든 변종 커버)
        guide_len       = 20,
        pam_position    = "3prime",
        cut_offset      = -3,
        model_path      = "/content/model.keras",
        model_input_len = 30,
    ),
    CasConfig(
        name            = "SaCas9",
        pam_iupac       = "NNGRRT",
        guide_len       = 21,
        pam_position    = "3prime",
        cut_offset      = -3,
        model_path      = "",
        model_input_len = 36,
    ),
    CasConfig(
        name            = "Cas12a",
        pam_iupac       = "TTTV",
        guide_len       = 23,
        pam_position    = "5prime",
        cut_offset      = 20,
        model_path      = "/content/model_cas12a.keras",
        model_input_len = 34,
    ),
]

_IUPAC_TABLE: dict[str, str] = {
    'A': 'A', 'T': 'T', 'G': 'G', 'C': 'C',
    'N': '[ATGC]',
    'R': '[AG]',
    'Y': '[CT]',
    'S': '[GC]',
    'W': '[AT]',
    'K': '[GT]',
    'M': '[AC]',
    'B': '[CGT]',
    'D': '[AGT]',
    'H': '[ACT]',
    'V': '[ACG]',
}

# SpCas9 9개 변종 이름 (predict.py와 동기화)
SPCAS9_VARIANTS = [
    'SpCas9(WT)', 'SpCas9-NG', 'VRQR', 'xCas9',
    'Sniper', 'SpCas9-HF1', 'eSpCas9(1.1)', 'HypaCas9', 'evoCas9'
]


# ════════════════════════════════════════════════
# 섹션 2. 데이터 구조
# ════════════════════════════════════════════════

@dataclass
class CandidateSite:
    """유효한 절단 후보 부위 하나의 정보"""
    cas_type:        str
    strand:          str
    pam_start:       int
    cut_pos:         int
    distance:        int
    guide_seq:       str
    pam_seq:         str
    model_input_seq: str   = ""
    encoded_seq:     list  = field(default_factory=list)
    raw_score:       float = 0.0
    final_score:     float = 0.0


@dataclass
class VariantResult:
    """
    [신규] SpCas9 변종 개별 예측 결과.
    하나의 CandidateSite에서 9개 변종 각각에 대해 생성됨.
    """
    cas_name:    str    # 예: 'SpCas9(WT)', 'SpCas9-NG', ...
    pam_seq:     str
    guide_seq:   str
    distance:    int
    strand:      str
    raw_score:   float
    final_score: float


@dataclass
class ScanResult:
    """전체 스캔 결과 컨테이너"""
    input_seq:      str
    center_idx:     int                       = 40
    sites:          list[CandidateSite]       = field(default_factory=list)
    # [신규] 변종별 예측 결과를 담는 flat 리스트 (predict.py에서 채워짐)
    variant_results: list[VariantResult]      = field(default_factory=list)

    def by_cas(self, cas_type: str) -> list[CandidateSite]:
        return [s for s in self.sites if s.cas_type == cas_type]

    def sorted_by_score(self) -> list[CandidateSite]:
        return sorted(self.sites, key=lambda s: s.final_score, reverse=True)

    def sorted_variant_results(self) -> list[VariantResult]:
        """[신규] 변종 결과를 final_score 내림차순으로 정렬"""
        return sorted(self.variant_results, key=lambda v: v.final_score, reverse=True)

    def __repr__(self) -> str:
        counts = {cfg.name: len(self.by_cas(cfg.name)) for cfg in CAS_CONFIGS}
        parts  = ", ".join(f"{k}={v}" for k, v in counts.items())
        return f"ScanResult({parts}, total={len(self.sites)})"


# ════════════════════════════════════════════════
# 섹션 3. 유틸리티 함수
# ════════════════════════════════════════════════

def one_hot_encode(seq: str) -> list:
    mapping = {
        'A': [1, 0, 0, 0], 'C': [0, 1, 0, 0],
        'G': [0, 0, 1, 0], 'T': [0, 0, 0, 1],
        'R': [0.5, 0, 0.5, 0],  'Y': [0, 0.5, 0, 0.5],
        'S': [0, 0.5, 0.5, 0],  'W': [0.5, 0, 0, 0.5],
        'K': [0, 0, 0.5, 0.5],  'M': [0.5, 0.5, 0, 0],
        'B': [0, 0.33, 0.33, 0.33], 'D': [0.33, 0, 0.33, 0.33],
        'H': [0.33, 0.33, 0, 0.33], 'V': [0.33, 0.33, 0.33, 0],
        'N': [0.25, 0.25, 0.25, 0.25]
    }
    return [mapping.get(b, [0,0,0,0]) for b in seq.upper()]

def iupac_to_regex(pam: str) -> str:
    try:
        return ''.join(_IUPAC_TABLE[b] for b in pam.upper())
    except KeyError as e:
        raise ValueError(f"알 수 없는 IUPAC 코드: {e}")

def reverse_complement(seq: str) -> str:
    table = str.maketrans("ACGTRYSWKMBDHVNacgtryswkmbdhvn",
                           "TGCAYRSWMKVHDBNtgcayrswmkvhdbn")
    return seq.translate(table)[::-1]

_IUPAC_BASES: dict[str, set[str]] = {
    'A': {'A'}, 'T': {'T'}, 'G': {'G'}, 'C': {'C'},
    'N': {'A','T','G','C'},
    'R': {'A','G'}, 'Y': {'C','T'}, 'S': {'G','C'},
    'W': {'A','T'}, 'K': {'G','T'}, 'M': {'A','C'},
    'B': {'C','G','T'}, 'D': {'A','G','T'},
    'H': {'A','C','T'}, 'V': {'A','C','G'},
}

def iupac_match(seq_char: str, pam_char: str) -> bool:
    return bool(_IUPAC_BASES[seq_char] & _IUPAC_BASES[pam_char])

def iupac_pam_search(seq: str, pam: str) -> list[tuple[int, int, str]]:
    matches = []
    pam_len = len(pam)
    for i in range(len(seq) - pam_len + 1):
        window = seq[i:i + pam_len]
        if all(iupac_match(window[j], pam[j]) for j in range(pam_len)):
            matches.append((i, i + pam_len, window))
    return matches

def validate_sequence(seq: str) -> str:
    seq = "".join(seq.upper().split())
    if len(seq) != 81:
        raise ValueError(
            f"입력 서열은 81bp여야 합니다. (현재: {len(seq)}bp)\n"
            "변이 위치 기준 앞뒤 40bp씩 총 81bp를 입력해 주세요."
        )
    VALID_IUPAC = set("ATGCNRYSWKMBDHV")
    invalid = set(seq) - VALID_IUPAC
    if invalid:
        raise ValueError(f"허용되지 않는 문자 포함: {invalid}")
    return seq

def gaussian_penalty(distance: int, sigma: float = 10.0) -> float:
    return math.exp(-(distance ** 2) / (2 * sigma ** 2))

def _wcswidth(s: str) -> int:
    return sum(2 if unicodedata.east_asian_width(c) in ('W', 'F') else 1 for c in s)

def _ljust_wide(s: str, width: int) -> str:
    return s + ' ' * max(width - _wcswidth(s), 0)

def dna_to_rna(seq: str) -> str:
    """
    출력 전용 T→U 변환 함수.
    내부 연산(PAM 탐색, 인코딩 등)에는 절대 사용하지 않음.
    """
    return seq.replace('T', 'U').replace('t', 'u')


# ════════════════════════════════════════════════
# 섹션 4-1. 개별 PAM 존재 여부 탐색 함수
# ════════════════════════════════════════════════

def has_spcas9_pam(seq: str) -> bool:
    """
    SpCas9 및 9가지 변종 PAM 존재 여부 확인.
    [수정] 2bp NG 패턴으로 탐색 (SpCas9-NG 포함)
    """
    seq = seq.upper()
    pam_re = re.compile(iupac_to_regex("NG"))
    return bool(pam_re.search(seq)) or bool(pam_re.search(reverse_complement(seq)))

def has_sacas9_pam(seq: str) -> bool:
    seq    = seq.upper()
    pam_re = re.compile(iupac_to_regex("NNGRRT"))
    return bool(pam_re.search(seq)) or bool(pam_re.search(reverse_complement(seq)))

def has_cas12a_pam(seq: str) -> bool:
    seq    = seq.upper()
    pam_re = re.compile(iupac_to_regex("TTTV"))
    return bool(pam_re.search(seq)) or bool(pam_re.search(reverse_complement(seq)))

def check_all_pams(seq: str) -> dict[str, bool]:
    return {
        "SpCas9": has_spcas9_pam(seq),
        "SaCas9": has_sacas9_pam(seq),
        "Cas12a": has_cas12a_pam(seq),
    }


# ════════════════════════════════════════════════
# 섹션 4-2. PAM 탐색 (공통 스캔 로직)
# ════════════════════════════════════════════════

def _calc_cut_pos(cfg: CasConfig, pam_start: int, pam_end: int,
                  strand: str, seq_len: int) -> int:
    if cfg.pam_position == "3prime":
        raw_cut = pam_start + cfg.cut_offset
    else:
        raw_cut = pam_end + cfg.cut_offset
    return raw_cut if strand == '+' else seq_len - raw_cut - 1

def extract_model_input_window(search_seq: str, pam_start: int, pam_end: int,
                                cfg: CasConfig) -> str:
    seq_len    = len(search_seq)
    window_len = cfg.model_input_len

    if cfg.pam_position == "3prime":
        # [수정] PAM이 2bp(NG)로 바뀌었으므로 윈도우 시작점도 그에 맞게 계산
        # model_input_len=30: guide(20bp) + PAM(2bp) + upstream(4bp) + downstream(4bp) 구성
        window_start = pam_start - cfg.guide_len - 4
        window_end   = window_start + window_len
    else:
        window_start = pam_start
        window_end   = window_start + window_len

    if window_start < 0 or window_end > seq_len:
        return ""

    return search_seq[window_start:window_end]

def scan_pam(seq: str, cfg: CasConfig,
             center: int = 40, max_dist: int = 15) -> list[CandidateSite]:
    sites:   list[CandidateSite] = []
    seq_len = len(seq)

    for strand, search_seq in [('+', seq), ('-', reverse_complement(seq))]:
        for pam_start, pam_end, matched_window in iupac_pam_search(search_seq, cfg.pam_iupac):

            if cfg.pam_position == "3prime":
                guide_start = pam_start - cfg.guide_len
                guide_end   = pam_start
                if guide_start < 0:
                    continue
            else:
                guide_start = pam_end
                guide_end   = pam_end + cfg.guide_len
                if guide_end > seq_len:
                    continue

            cut_pos  = _calc_cut_pos(cfg, pam_start, pam_end, strand, seq_len)
            distance = abs(cut_pos - center)
            if distance > max_dist:
                continue

            model_seq = extract_model_input_window(search_seq, pam_start, pam_end, cfg)

            sites.append(CandidateSite(
                cas_type        = cfg.name,
                strand          = strand,
                pam_start       = pam_start,
                cut_pos         = cut_pos,
                distance        = distance,
                guide_seq       = search_seq[guide_start:guide_end],
                pam_seq         = matched_window,
                model_input_seq = model_seq,
                encoded_seq     = one_hot_encode(model_seq) if model_seq else [],
            ))

    return sites


def print_pam_analysis(input_dna: str) -> ScanResult:
    clean_seq   = validate_sequence(input_dna)
    all_results = ScanResult(input_seq=clean_seq)

    print(f"\n🔍 DNA 서열 분석 시작 (길이: {len(clean_seq)}bp)")
    print("-" * 60)

    for cfg in CAS_CONFIGS:
        found_sites = scan_pam(clean_seq, cfg)
        all_results.sites.extend(found_sites)

        print(f"[{cfg.name}] 탐색 결과:")
        if not found_sites:
            print("  - 발견된 PAM 서열이 없습니다.")
        for site in found_sites:
            # ── DNA 표기 (기존 출력 유지) ──────────────────────────────────────
            print(f"  · 가닥: {site.strand} | PAM(DNA): {site.pam_seq} | 위치: {site.pam_start} | 가이드(DNA): {site.guide_seq}")
            # ── RNA 표기 (T→U 변환, 신규 추가) ────────────────────────────────
            print(f"           PAM(RNA): {dna_to_rna(site.pam_seq)}        가이드(RNA): {dna_to_rna(site.guide_seq)}")
        print("-" * 60)

    print("\n📊 [PAM 탐색 최종 요약]")
    print("+" + "-"*15 + "+" + "-"*15 + "+")
    print(f"| {'Cas 모델':^13} | {'발견된 개수':^11} |")
    print("+" + "-"*15 + "+" + "-"*15 + "+")
    for cfg in CAS_CONFIGS:
        count = len(all_results.by_cas(cfg.name))
        print(f"| {cfg.name:<13} | {count:^13} |")
    print("+" + "-"*15 + "+" + "-"*15 + "+")
    print(f"| {'합계':<13} | {len(all_results.sites):^13} |")
    print("+" + "-"*15 + "+" + "-"*15 + "+")

    return all_results

Overwriting /content/pam_scanner.py


# predict.py

In [9]:
%%writefile /content/predict.py
"""
predict.py
──────────
모델 로딩 및 절단 효율 예측 전담 모듈.

[변경사항]
- run_prediction: SpCas9 사이트에서 9개 변종을 각각 VariantResult로 생성해
  all_results.variant_results 리스트에 저장.
- SaCas9, Cas12a는 기존대로 site.raw_score / site.final_score에 저장.
"""

import numpy as np
import tensorflow as tf
import torch
from pam_scanner import gaussian_penalty, ScanResult, VariantResult
from adapters import OneHotAdapter
from backbones import CNN_RNN_Backbone, IntegratedPredictor

# SpCas9 변종 이름 (모델 출력 인덱스 순서와 일치)
SPCAS9_VARIANTS = [
    'SpCas9(WT)', 'SpCas9-NG', 'VRQR', 'xCas9',
    'Sniper', 'SpCas9-HF1', 'eSpCas9(1.1)', 'HypaCas9', 'evoCas9'
]

MODEL_SCORE_SCALE = {
    'SpCas9': 100,
    'SaCas9': 100,
    'Cas12a': 1,
}


def load_models():
    model        = tf.keras.models.load_model("/content/model.keras")
    model_cas12a = tf.keras.models.load_model("/content/model_cas12a.keras")

    weight_files = [
        "/content/best_model_fold1.pth",
        "/content/best_model_fold2.pth",
        "/content/best_model_fold3.pth",
        "/content/best_model_fold4.pth",
        "/content/best_model_fold5.pth",
    ]

    models_sa = []
    for w_path in weight_files:
        adapter  = OneHotAdapter(input_dim=4, hidden_dim=128)
        backbone = CNN_RNN_Backbone(input_dim=128, lstm_hidden=128, dropout=0.0)
        m        = IntegratedPredictor(adapter=adapter, backbone=backbone)
        m.load_state_dict(torch.load(w_path, map_location="cpu"))
        m.eval()
        models_sa.append(m)

    return model, models_sa, model_cas12a


def one_hot_encode_dna(sequence: str) -> np.ndarray:
    mapping = {
        'A': [1,0,0,0], 'C': [0,1,0,0],
        'G': [0,0,1,0], 'T': [0,0,0,1],
    }
    return np.array([mapping.get(b, [0,0,0,0]) for b in sequence.upper()])


def run_prediction(all_results: ScanResult, model, models_sa, model_cas12a) -> ScanResult:
    """
    모든 CandidateSite에 대해 모델 예측을 수행.

    SpCas9:
      - 케라스 모델이 9개 변종 점수를 동시에 반환 (preds 리스트)
      - 각 변종을 VariantResult로 만들어 all_results.variant_results에 추가
      - site 자체에는 9개 변종 중 최고 점수만 반영 (순위표 통합 표시용)

    SaCas9 / Cas12a:
      - 기존대로 site.raw_score / site.final_score에 저장
      - 동시에 VariantResult도 생성해 variant_results에 추가 (통합 정렬용)
    """
    all_results.variant_results = []  # 초기화

    for site in all_results.sites:
        if not site.model_input_seq:
            continue

        X_np    = np.expand_dims(one_hot_encode_dna(site.model_input_seq), axis=0)
        penalty = gaussian_penalty(site.distance)

        # ── SpCas9: 9개 변종 개별 처리 ──────────────────────────────
        if site.cas_type == 'SpCas9':
            # model.predict()는 [output0, output1, ..., output8] 형태 반환
            # output_i.shape = (1, 1) → [0][0]으로 float 추출
            preds = model.predict(X_np, verbose=0)

            best_score = -1.0
            for i, variant_name in enumerate(SPCAS9_VARIANTS):
                raw         = float(preds[i][0][0])
                final_score = raw * penalty * MODEL_SCORE_SCALE['SpCas9']

                all_results.variant_results.append(VariantResult(
                    cas_name    = variant_name,
                    pam_seq     = site.pam_seq,
                    guide_seq   = site.guide_seq,
                    distance    = site.distance,
                    strand      = site.strand,
                    raw_score   = raw,
                    final_score = final_score,
                ))

                if final_score > best_score:
                    best_score   = final_score
                    best_raw     = raw

            # site에는 9개 중 최고값 저장 (app.py의 통합 순위표에 사용)
            site.raw_score   = best_raw
            site.final_score = best_score

        # ── SaCas9: PyTorch 5-Fold 앙상블 ───────────────────────────
        elif site.cas_type == 'SaCas9':
            X_tensor = torch.tensor(X_np, dtype=torch.float32)
            scores   = []
            with torch.no_grad():
                for m in models_sa:
                    scores.append(float(m(X_tensor).item()))
            raw         = sum(scores) / len(scores)
            final_score = raw * penalty * MODEL_SCORE_SCALE['SaCas9']

            site.raw_score   = raw
            site.final_score = final_score

            all_results.variant_results.append(VariantResult(
                cas_name    = 'SaCas9',
                pam_seq     = site.pam_seq,
                guide_seq   = site.guide_seq,
                distance    = site.distance,
                strand      = site.strand,
                raw_score   = raw,
                final_score = final_score,
            ))

        # ── Cas12a ───────────────────────────────────────────────────
        elif site.cas_type == 'Cas12a':
            raw         = float(model_cas12a.predict(X_np, verbose=0)[0][0])
            final_score = raw * penalty * MODEL_SCORE_SCALE['Cas12a']

            site.raw_score   = raw
            site.final_score = final_score

            all_results.variant_results.append(VariantResult(
                cas_name    = 'Cas12a',
                pam_seq     = site.pam_seq,
                guide_seq   = site.guide_seq,
                distance    = site.distance,
                strand      = site.strand,
                raw_score   = raw,
                final_score = final_score,
            ))

    return all_results


Overwriting /content/predict.py


# app.py 파일 저장

In [10]:
%%writefile /content/app.py
import streamlit as st
import pandas as pd
from pam_scanner import print_pam_analysis, validate_sequence, CAS_CONFIGS, SPCAS9_VARIANTS, dna_to_rna
from predict import load_models, run_prediction

st.set_page_config(page_title="CRISPR PAM Scanner", layout="wide")
st.title("🧬 CRISPR PAM Scanner")
st.markdown("81bp DNA 서열을 입력하면 PAM 탐색 및 절단 효율을 예측합니다.")


@st.cache_resource
def get_models():
    return load_models()

model, models_sa, model_cas12a = get_models()

if 'all_results' not in st.session_state:
    st.session_state.all_results = None
if 'detail_filtered' not in st.session_state:
    st.session_state.detail_filtered = None

dna_input = st.text_input("81bp DNA 서열 입력", placeholder="ATCG...")
run_btn   = st.button("🔍 분석 시작")

if run_btn and dna_input:
    try:
        validate_sequence(dna_input)

        with st.spinner("PAM 탐색 중..."):
            all_results = print_pam_analysis(dna_input)

        with st.spinner("모델 예측 중..."):
            all_results = run_prediction(all_results, model, models_sa, model_cas12a)

        st.session_state.all_results     = all_results
        st.session_state.detail_filtered = None

    except ValueError as e:
        st.error(f"❌ 오류: {e}")

if st.session_state.all_results is not None:
    all_results = st.session_state.all_results

    st.subheader("📊 PAM 탐색 결과")
    col1, col2, col3 = st.columns(3)
    for col, cfg in zip([col1, col2, col3], CAS_CONFIGS):
        sites = all_results.by_cas(cfg.name)
        col.metric(label=cfg.name, value=f"{len(sites)}개 발견")

    st.subheader("🤖 절단 효율 예측 결과")

    tab_best, tab_sp, tab_others, tab_detail = st.tabs([
        "🏆 Best Candidates",
        "🔬 SpCas9 변종 히트맵",
        "🧪 SaCas9 / Cas12a",
        "📋 전체 상세 보기",
    ])

    # ════════════════════════════════════════════════════════
    # 탭1: Best Candidates
    # ════════════════════════════════════════════════════════
    with tab_best:
        st.markdown(
            "각 가이드 서열에서 **가장 높은 효율을 보인 변종 1개**만 대표로 표시합니다. "
            "자세한 변종별 비교는 다른 탭을 확인하세요."
        )
        if not all_results.variant_results:
            st.info("예측 결과가 없습니다.")
        else:
            guide_best: dict[str, object] = {}
            for vr in all_results.variant_results:
                key = vr.guide_seq
                if key not in guide_best or vr.final_score > guide_best[key].final_score:
                    guide_best[key] = vr

            sorted_best = sorted(guide_best.values(), key=lambda v: v.final_score, reverse=True)
            total_best  = len(sorted_best)
            display_best = sorted_best[:20]   # 최대 20개

            if total_best > 20:
                st.caption(f"상위 20개만 표시합니다. (전체 {total_best}개) 전체 결과는 '전체 상세 보기' 탭을 이용하세요.")

            rows_best = []
            for rank, vr in enumerate(display_best, start=1):
                rows_best.append({
                    '순위':              rank,
                    '최적 Cas 변종':      vr.cas_name,
                    'PAM':               vr.pam_seq,
                    '가이드 서열 (DNA)':  vr.guide_seq,
                    '가이드 서열 (RNA)':  dna_to_rna(vr.guide_seq),
                    '가닥':              vr.strand,
                    '거리 (bp)':         vr.distance,
                    '최종 효율 (%)':      round(vr.final_score, 2),
                })

            df_best = pd.DataFrame(rows_best)
            st.dataframe(
                df_best.style.background_gradient(
                    subset=['최종 효율 (%)'], cmap='RdYlGn', vmin=0, vmax=100
                ),
                use_container_width=True,
                hide_index=True,
                height=36 * (len(df_best) + 1) + 3,
            )

    # ════════════════════════════════════════════════════════
    # 탭2: SpCas9 변종 히트맵
    # ════════════════════════════════════════════════════════
    with tab_sp:
        sp_results = [vr for vr in all_results.variant_results
                      if vr.cas_name in SPCAS9_VARIANTS]

        if not sp_results:
            st.info("SpCas9 PAM 사이트가 발견되지 않았습니다.")
        else:
            st.markdown(
                "행: 가이드 서열 / 열: SpCas9 변종. "
                "**색이 밝을수록(노랑/주황) 절단 효율이 높고, 진할수록(빨강) 낮습니다.**"
            )
            pivot_rows = []
            for vr in sp_results:
                # 옵션F: RNA 서열 + 라벨 명시 메타정보
                label = (
                    f"{dna_to_rna(vr.guide_seq)}  dist:{vr.distance}  PAM:{vr.pam_seq}  strand:{vr.strand}"
                )
                pivot_rows.append({
                    'guide_label': label,
                    'variant':     vr.cas_name,
                    'score':       round(vr.final_score, 2),
                })

            df_pivot = (
                pd.DataFrame(pivot_rows)
                .pivot_table(index='guide_label', columns='variant',
                             values='score', aggfunc='max')
                .reindex(columns=[v for v in SPCAS9_VARIANTS
                                  if v in pd.DataFrame(pivot_rows)['variant'].unique()])
                .assign(_max=lambda d: d.max(axis=1))
                .sort_values('_max', ascending=False)
                .drop(columns='_max')
            )

            total_sp = len(df_pivot)
            if total_sp > 20:
                st.caption(f"상위 20개만 표시합니다. (전체 {total_sp}개) 전체 결과는 '전체 상세 보기' 탭을 이용하세요.")
            df_pivot = df_pivot.head(20)

            # 모든 열(변종 점수 열) 너비를 100px으로 균등하게 설정
            col_config = {col: st.column_config.NumberColumn(col, width=100)
                          for col in df_pivot.columns}

            st.dataframe(
                df_pivot.style.background_gradient(cmap='RdYlGn', vmin=0, vmax=100)
                        .format("{:.1f}"),
                use_container_width=True,
                height=36 * (len(df_pivot) + 1) + 3,
                column_config=col_config,
            )

    # ════════════════════════════════════════════════════════
    # 탭3: SaCas9 / Cas12a
    # ════════════════════════════════════════════════════════
    with tab_others:
        other_results = [vr for vr in all_results.variant_results
                         if vr.cas_name in ('SaCas9', 'Cas12a')]

        if not other_results:
            st.info("SaCas9 / Cas12a PAM 사이트가 발견되지 않았습니다.")
        else:
            rows_ot = []
            for i, vr in enumerate(
                sorted(other_results, key=lambda v: v.final_score, reverse=True), start=1
            ):
                rows_ot.append({
                    '순위':              i,
                    'Cas 종류':          vr.cas_name,
                    'PAM':               vr.pam_seq,
                    '가이드 서열 (DNA)':  vr.guide_seq,
                    '가이드 서열 (RNA)':  dna_to_rna(vr.guide_seq),  # RNA 추가
                    '가닥':              vr.strand,
                    '거리 (bp)':         vr.distance,
                    'Raw Score':         round(vr.raw_score, 4),
                    '최종 효율 (%)':      round(vr.final_score, 2),
                })

            st.dataframe(
                pd.DataFrame(rows_ot).style.background_gradient(
                    subset=['최종 효율 (%)'], cmap='RdYlGn', vmin=0, vmax=100
                ),
                use_container_width=True,
                hide_index=True,
            )

    # ════════════════════════════════════════════════════════
    # 탭4: 전체 상세 보기
    # ════════════════════════════════════════════════════════
    with tab_detail:
        st.markdown("원하는 조건을 설정한 뒤 **검색 버튼**을 눌러 결과를 확인하세요.")

        all_vr = all_results.sorted_variant_results()

        if not all_vr:
            st.info("예측 결과가 없습니다.")
        else:
            all_cas_names = sorted(set(vr.cas_name for vr in all_vr))

            for name in all_cas_names:
                if f"chk_{name}" not in st.session_state:
                    st.session_state[f"chk_{name}"] = True

            st.markdown("#### 🔧 필터 설정")
            fc1, fc2 = st.columns([1, 1])

            with fc1:
                st.markdown("**Cas 변종 선택**")
                tc1, tc2 = st.columns(2)

                if tc1.button("전체 선택", key="sel_all"):
                    for name in all_cas_names:
                        st.session_state[f"chk_{name}"] = True
                    st.rerun()

                if tc2.button("전체 해제", key="desel_all"):
                    for name in all_cas_names:
                        st.session_state[f"chk_{name}"] = False
                    st.rerun()

                with st.container(height=280, border=True):
                    for name in all_cas_names:
                        st.checkbox(name, key=f"chk_{name}")

            with fc2:
                st.markdown("**효율 범위 및 거리 설정**")
                st.markdown(" ")

                eff_range = st.slider(
                    "최종 효율 범위 (%)",
                    min_value=0.0, max_value=100.0,
                    value=(0.0, 100.0), step=0.5,
                    format="%.1f%%",
                    key="slider_eff",
                )
                st.caption(f"선택 범위: {eff_range[0]:.1f}% ~ {eff_range[1]:.1f}%")

                st.markdown(" ")

                dist_range = st.slider(
                    "거리 범위 (bp)",
                    min_value=0, max_value=15,
                    value=(0, 15), step=1,
                    format="%d bp",
                    key="slider_dist",
                )
                st.caption(f"변이로부터 {dist_range[0]} ~ {dist_range[1]} bp 이내")

            st.markdown(" ")

            if st.button("🔍 검색", type="primary", key="detail_search"):
                sel_cas          = [n for n in all_cas_names if st.session_state[f"chk_{n}"]]
                min_eff, max_eff = eff_range
                min_dist, max_dist = dist_range

                st.session_state.detail_filtered = [
                    vr for vr in all_vr
                    if vr.cas_name in sel_cas
                    and min_eff <= vr.final_score <= max_eff
                    and min_dist <= vr.distance <= max_dist
                ]

            if st.session_state.detail_filtered is not None:
                filtered = st.session_state.detail_filtered
                st.caption(f"검색 결과: {len(filtered)}개 / 전체 {len(all_vr)}개")

                if filtered:
                    rows_detail = []
                    for i, vr in enumerate(filtered, start=1):
                        rows_detail.append({
                            '순위':              i,
                            'Cas 변종':          vr.cas_name,
                            'PAM':               vr.pam_seq,
                            '가이드 서열 (DNA)':  vr.guide_seq,
                            '가이드 서열 (RNA)':  dna_to_rna(vr.guide_seq),  # RNA 추가
                            '가닥':              vr.strand,
                            '거리 (bp)':         vr.distance,
                            'Raw Score':         round(vr.raw_score, 4),
                            '최종 효율 (%)':      round(vr.final_score, 2),
                        })

                    df_detail = pd.DataFrame(rows_detail)
                    st.dataframe(
                        df_detail.style.background_gradient(
                            subset=['최종 효율 (%)'], cmap='RdYlGn', vmin=0, vmax=100
                        ),
                        use_container_width=True,
                        hide_index=True,
                    )

                    csv = df_detail.to_csv(index=False).encode('utf-8-sig')
                    st.download_button(
                        label="⬇️ CSV 다운로드",
                        data=csv,
                        file_name="crispr_prediction_results.csv",
                        mime="text/csv",
                        key="csv_download",
                    )
                else:
                    st.warning("필터 조건에 맞는 결과가 없습니다.")


Overwriting /content/app.py


# Streamlit 실행

In [13]:
from pyngrok import ngrok
import subprocess
import os
import requests

# ngrok API로 기존 터널 직접 삭제
try:
    tunnels = requests.get("http://localhost:4040/api/tunnels").json()
    for tunnel in tunnels.get("tunnels", []):
        requests.delete(f"http://localhost:4040/api/tunnels/{tunnel['name']}")
    print("✅ 기존 터널 삭제 완료")
except:
    print("ℹ️ 삭제할 기존 터널 없음")

# 프로세스 정리
ngrok.kill()
os.system("pkill -f 'streamlit run' 2>/dev/null")

import time
time.sleep(2)  # 완전히 종료될 때까지 대기

ngrok.set_auth_token("3DIe6sUE6TpnmZzlxT7FH6DvWJa_5dt2N3s7W2mwVGfT2ahZH")

process = subprocess.Popen(["streamlit", "run", "app.py",
                            "--server.port=8501",
                            "--server.headless=true"])

time.sleep(3)  # Streamlit 뜰 때까지 대기

public_url = ngrok.connect(8501)
print(f"✅ 접속 URL: {public_url}")

✅ 기존 터널 삭제 완료
✅ 접속 URL: NgrokTunnel: "https://emcee-slip-grit.ngrok-free.dev" -> "http://localhost:8501"


# pam_scanner.py


In [ ]:
"""
pam_scanner.py
──────────────
PAM 서열 탐색 전담 모듈.

포함 내용:
  - CasConfig        : Cas 단백질 설정 (PAM, 가이드 길이, 절단 위치 등)
  - CandidateSite    : 개별 절단 후보 데이터 클래스
  - ScanResult       : 스캔 결과 컨테이너
  - 유틸리티 함수    : iupac_to_regex, reverse_complement, validate_sequence,
                       gaussian_penalty, _wcswidth, _ljust_wide
  - PAM 탐색 함수    : has_spcas9_pam, has_sacas9_pam, has_cas12a_pam,
                       check_all_pams, scan_pam
"""

import re
import math
import unicodedata
from dataclasses import dataclass, field
from typing import Optional
import torch


# ════════════════════════════════════════════════
# 섹션 1. 설정
# ════════════════════════════════════════════════

@dataclass(frozen=True)  # 객체 읽기 전용, 해시 가능
class CasConfig:
    """
    Cas 단백질 한 종류의 PAM/절단 규칙 정의.
    새 단백질 추가 시 이 클래스 인스턴스만 하나 더 만들면 됨.
    """
    name:               str   # 유전자 가위 이름
    pam_iupac:          str   # PAM 서열 규칙
    guide_len:          int   # 모델이 인식할 서열의 길이 결정
    pam_position:       str   # '3prime' | '5prime' / PAM 위치
    cut_offset:         int   # DNA가 잘리는 위치
    model_path:         str   # 효율 예측할 때 어떤 모델 파일(.onnx)을 사용할 것인지 정해주는 경로
    model_input_len:    int

# 3가지 유전자 가위의 구체적인 명세서
CAS_CONFIGS: list[CasConfig] = [
    CasConfig(
        name         = "SpCas9",
        pam_iupac    = "NGN",
        guide_len    = 20,
        pam_position = "3prime",
        cut_offset   = -3,
        model_path   = "/content/model.keras",
        model_input_len = 30,
    ),
    CasConfig(
        name         = "SaCas9",
        pam_iupac    = "NNGRRT",
        guide_len    = 21,
        pam_position = "3prime",
        cut_offset   = -3,
        model_path   = "/content/model_sa.keras",
        model_input_len = 36,
    ),
    CasConfig(
        name         = "Cas12a",
        pam_iupac    = "TTTV",
        guide_len    = 23,
        pam_position = "5prime",
        cut_offset   = 20,
        model_path   = "/content/model_cas12a.keras",
        model_input_len = 34,
    ),
]

# 이후에 추가 모델의 PAM 서열을 인식해야 할 경우를 대비해 모든 IUPAC 넣어놓음
_IUPAC_TABLE: dict[str, str] = {
    'A': 'A', 'T': 'T', 'G': 'G', 'C': 'C',
    'N': '[ATGC]',
    'R': '[AG]',
    'Y': '[CT]',
    'S': '[GC]',
    'W': '[AT]',
    'K': '[GT]',
    'M': '[AC]',
    'B': '[CGT]',
    'D': '[AGT]',
    'H': '[ACT]',
    'V': '[ACG]',
}

# ════════════════════════════════════════════════
# 섹션 2. 데이터 구조
# ════════════════════════════════════════════════

@dataclass
class CandidateSite:
    """유효한 절단 후보 부위 하나의 정보"""
    cas_type:    str            # 가위 종류
    strand:      str            # 가닥 방향(정방향 +, 역방향 -)
    pam_start:   int            # PAM 시작점(인덱스 형태)
    cut_pos:     int            # 절단 위치
    distance:    int            # 변이와의 거리
    guide_seq:   str            # gRNA 서열 (사용자에게 보여줄 가이드 서열, 20/21/23bp)
    pam_seq:     str            # 발견된 PAM 서열
    model_input_seq: str = ""   # AI 모델 입력용 서열(30bp)
    encoded_seq: list = field(default_factory=list)  # 원핫 인코딩 데이터 저장용
    raw_score:   float = 0.0    # 순수 절단 효율 (0~1)
    final_score: float = 0.0    # 거리 패널티 적용 후 %로 변환한 최종 값


@dataclass
class ScanResult:
    """전체 스캔 결과 컨테이너"""
    input_seq:  str                                                         # 사용자가 입력한 81bp DNA 서열
    center_idx: int                        = 40                             # 타겟 변위의 위치
    sites:      list[CandidateSite]        = field(default_factory=list)    # 발견된 모든 CandidateSite들

    def by_cas(self, cas_type: str) -> list[CandidateSite]:
        """필터링 - 특정 가위 종류만 골라냄"""
        return [s for s in self.sites if s.cas_type == cas_type]

    def sorted_by_score(self) -> list[CandidateSite]:
        """정렬 - 모든 후보의 점수를 내림차순으로 정렬"""
        return sorted(self.sites, key=lambda s: s.final_score, reverse=True)

    def __repr__(self) -> str:
        """요약 출력 - 가위별로 찾은 PAM 개수"""
        counts = {cfg.name: len(self.by_cas(cfg.name)) for cfg in CAS_CONFIGS}
        parts  = ", ".join(f"{k}={v}" for k, v in counts.items())
        return f"ScanResult({parts}, total={len(self.sites)})"


# ════════════════════════════════════════════════
# 섹션 3. 유틸리티 함수
# ════════════════════════════════════════════════

def one_hot_encode(seq: str) -> list:
    """DNA 서열을 4xL 형태의 원핫 인코딩으로 변환"""
    mapping = {
        'A': [1, 0, 0, 0],
        'C': [0, 1, 0, 0],
        'G': [0, 0, 1, 0],
        'T': [0, 0, 0, 1],
        'R': [0.5, 0, 0.5, 0],    # A or G
        'Y': [0, 0.5, 0, 0.5],    # C or T
        'S': [0, 0.5, 0.5, 0],    # G or C
        'W': [0.5, 0, 0, 0.5],    # A or T
        'K': [0, 0, 0.5, 0.5],    # G or T
        'M': [0.5, 0.5, 0, 0],    # A or C
        'B': [0, 0.33, 0.33, 0.33],
        'D': [0.33, 0, 0.33, 0.33],
        'H': [0.33, 0.33, 0, 0.33],
        'V': [0.33, 0.33, 0.33, 0],
        'N': [0.25, 0.25, 0.25, 0.25]
    }
    return [mapping.get(b, [0,0,0,0]) for b in seq.upper()]

def iupac_to_regex(pam: str) -> str:
    """IUPAC PAM 서열 → 파이썬 정규표현식 문자열"""
    try:
        return ''.join(_IUPAC_TABLE[b] for b in pam.upper())
    except KeyError as e:
        raise ValueError(f"알 수 없는 IUPAC 코드: {e}")

def reverse_complement(seq: str) -> str:
    """DNA 서열의 역상보(Reverse Complement) 반환 (IUPAC 심볼 포함)"""
    table = str.maketrans("ACGTRYSWKMBDHVNacgtryswkmbdhvn",
                           "TGCAYRSWMKVHDBNtgcayrswmkvhdbn")
    return seq.translate(table)[::-1]


# IUPAC 심볼별 허용 염기 집합
_IUPAC_BASES: dict[str, set[str]] = {
    'A': {'A'}, 'T': {'T'}, 'G': {'G'}, 'C': {'C'},
    'N': {'A','T','G','C'},
    'R': {'A','G'}, 'Y': {'C','T'}, 'S': {'G','C'},
    'W': {'A','T'}, 'K': {'G','T'}, 'M': {'A','C'},
    'B': {'C','G','T'}, 'D': {'A','G','T'},
    'H': {'A','C','T'}, 'V': {'A','C','G'},
}

def iupac_match(seq_char: str, pam_char: str) -> bool:
    """
    입력 서열의 문자 하나와 PAM의 IUPAC 심볼이 호환되는지 확인.
    양쪽 모두 IUPAC일 수 있으므로 허용 염기 집합의 교집합으로 판단.
    예) seq_char='R'(A or G), pam_char='N'(A/T/G/C) → 교집합 {A,G} → True
    """
    return bool(_IUPAC_BASES[seq_char] & _IUPAC_BASES[pam_char])


def iupac_pam_search(seq: str, pam: str) -> list[tuple[int, int, str]]:
    """
    입력 서열에서 PAM 패턴을 IUPAC 호환 방식으로 탐색.
    정규식 대신 위치별로 직접 비교하므로 입력 서열에 IUPAC 심볼이 있어도 동작.

    Returns
    -------
    list of (start, end, matched_window)
    """
    matches = []
    pam_len = len(pam)
    for i in range(len(seq) - pam_len + 1): # 윈도우가 전체 서열을 벗어나지 않도록 반복 범위 지정
        window = seq[i:i + pam_len] # 현재 인덱스에서 PAM 길이만큼 서열 잘라내어 window 변수에 저장
        if all(iupac_match(window[j], pam[j]) for j in range(pam_len)):
            matches.append((i, i + pam_len, window))
    return matches

def validate_sequence(seq: str) -> str:
    """입력 서열 유효성 검사 (81bp 고정, ATGCN 허용)"""
    seq = "".join(seq.upper().split())
    if len(seq) != 81:
        raise ValueError(
            f"입력 서열은 81bp여야 합니다. (현재: {len(seq)}bp)\n"
            "변이 위치 기준 앞뒤 40bp씩 총 81bp를 입력해 주세요."
        )
    VALID_IUPAC = set("ATGCNRYSWKMBDHV")  # 전체 IUPAC 허용
    invalid = set(seq) - VALID_IUPAC
    if invalid:
        raise ValueError(f"허용되지 않는 문자 포함: {invalid}")
    return seq


def gaussian_penalty(distance: int, sigma: float = 10.0) -> float:
    """거리 기반 가우시안 가중치 (거리 멀수록 0 수렴)
    sigma 값 줄이면 감점 폭 커짐"""
    return math.exp(-(distance ** 2) / (2 * sigma ** 2))


def _wcswidth(s: str) -> int:
    """터미널 실제 출력 폭 계산 (한글·전각문자=2, 그 외=1)"""
    return sum(2 if unicodedata.east_asian_width(c) in ('W', 'F') else 1 for c in s)


def _ljust_wide(s: str, width: int) -> str:
    """터미널 폭 기준 왼쪽 정렬 패딩 (한글 포함 문자열도 정확히 맞춤)"""
    return s + ' ' * max(width - _wcswidth(s), 0)


# ════════════════════════════════════════════════
# 섹션 4-1. 개별 PAM 존재 여부 탐색 함수
# ════════════════════════════════════════════════

def has_spcas9_pam(seq: str) -> bool:
    """
    서열 내 SpCas9 및 9가지 변종 PAM(NGN, GAA, GAT) 존재 여부 확인.
    """
    seq = seq.upper()
    pam_pattern = iupac_to_regex("NGN") + "|" + iupac_to_regex("GAA") + "|" + iupac_to_regex("GAT")
    pam_re = re.compile(pam_pattern)
    return bool(pam_re.search(seq)) or bool(pam_re.search(reverse_complement(seq)))


def has_sacas9_pam(seq: str) -> bool:
    """
    서열 내 SaCas9 PAM(NNGRRT) 존재 여부 확인.
    양쪽 가닥 모두 탐색.
    """
    seq    = seq.upper()
    pam_re = re.compile(iupac_to_regex("NNGRRT"))
    return bool(pam_re.search(seq)) or bool(pam_re.search(reverse_complement(seq)))


def has_cas12a_pam(seq: str) -> bool:
    """
    서열 내 Cas12a PAM(TTTV, V=A/C/G) 존재 여부 확인.
    양쪽 가닥 모두 탐색.
    """
    seq    = seq.upper()
    pam_re = re.compile(iupac_to_regex("TTTV"))
    return bool(pam_re.search(seq)) or bool(pam_re.search(reverse_complement(seq)))


def check_all_pams(seq: str) -> dict[str, bool]:
    """
    세 가지 Cas 단백질의 PAM 존재 여부를 한 번에 반환.

    Returns
    -------
    dict  예: {"SpCas9": True, "SaCas9": False, "Cas12a": True}
    """
    return {
        "SpCas9": has_spcas9_pam(seq),
        "SaCas9": has_sacas9_pam(seq),
        "Cas12a": has_cas12a_pam(seq),
    }


# ════════════════════════════════════════════════
# 섹션 4-2. PAM 탐색 (공통 스캔 로직)
# ════════════════════════════════════════════════

def _calc_cut_pos(cfg: CasConfig, pam_start: int, pam_end: int,
                  strand: str, seq_len: int) -> int:
    """절단 위치(원본 서열 기준) 계산. 역가닥은 좌표 변환."""
    if cfg.pam_position == "3prime":    # SpCas9, SaCas9는 PAM이 가이드 서열 뒤에 존재
        raw_cut = pam_start + cfg.cut_offset
    else:                               # Cas12a는 PAM이 가이드 서열 앞에 존재
        raw_cut = pam_end + cfg.cut_offset
    return raw_cut if strand == '+' else seq_len - raw_cut - 1

def extract_model_input_window(search_seq: str, pam_start: int, pam_end: int,
                                cfg: CasConfig) -> str:
    seq_len      = len(search_seq)
    window_len   = cfg.model_input_len  # ← 고정 30 대신 동적으로

    if cfg.pam_position == "3prime":
        window_start = pam_start - cfg.guide_len - 4
        window_end   = window_start + window_len
    else:
        window_start = pam_start
        window_end   = window_start + window_len

    if window_start < 0 or window_end > seq_len:
        return ""

    return search_seq[window_start:window_end]

def scan_pam(seq: str, cfg: CasConfig,
             center: int = 40, max_dist: int = 15) -> list[CandidateSite]:
    """단일 CasConfig 기준으로 양쪽 가닥에서 PAM을 탐색.
    입력 서열에 IUPAC 심볼이 포함되어 있어도 정상 동작."""
    sites:  list[CandidateSite] = []
    seq_len = len(seq)

    for strand, search_seq in [('+', seq), ('-', reverse_complement(seq))]:
        for pam_start, pam_end, matched_window in iupac_pam_search(search_seq, cfg.pam_iupac):

            if cfg.pam_position == "3prime":            # SpCas9, SaCas9
                guide_start = pam_start - cfg.guide_len
                guide_end   = pam_start
                if guide_start < 0:
                    continue
            else:                                       # Cas12a
                guide_start = pam_end
                guide_end   = pam_end + cfg.guide_len
                if guide_end > seq_len:
                    continue

            cut_pos  = _calc_cut_pos(cfg, pam_start, pam_end, strand, seq_len)
            distance = abs(cut_pos - center)
            if distance > max_dist:
                continue

            model_seq = extract_model_input_window(search_seq, pam_start, pam_end, cfg)  # ← 먼저 추출

            sites.append(CandidateSite(
                cas_type        = cfg.name,
                strand          = strand,
                pam_start       = pam_start,
                cut_pos         = cut_pos,
                distance        = distance,
                guide_seq       = search_seq[guide_start:guide_end],
                pam_seq         = matched_window,
                model_input_seq = model_seq,
                encoded_seq     = one_hot_encode(model_seq) if model_seq else [],
            ))

    return sites


def print_pam_analysis(input_dna: str) -> ScanResult:
    clean_seq = validate_sequence(input_dna)

    # 2. 결과 저장을 위한 컨테이너
    all_results = ScanResult(input_seq=clean_seq)

    # 3. 모든 Cas 설정에 대해 스캔 수행
    print(f"\n🔍 DNA 서열 분석 시작 (길이: {len(clean_seq)}bp)")
    print("-" * 60)

    for cfg in CAS_CONFIGS:
        found_sites = scan_pam(clean_seq, cfg)
        all_results.sites.extend(found_sites)

        # 상세 결과 출력 (터미널)
        print(f"[{cfg.name}] 탐색 결과:")
        if not found_sites:
            print("  - 발견된 PAM 서열이 없습니다.")
        for site in found_sites:
            print(f"  · 가닥: {site.strand} | PAM: {site.pam_seq} | 위치: {site.pam_start} | 가이드: {site.guide_seq}")
        print("-" * 60)

    # 4. 요약 표 출력
    print("\n📊 [PAM 탐색 최종 요약]")
    print("+---------------+---------------+")
    print("|    Cas 모델     |   발견된 개수    |")
    print("+---------------+---------------+")

    # 각 Cas 종류별 탐색 개수 계산
    sp_cnt = sum(1 for s in all_results.sites if s.cas_type == 'SpCas9')
    sa_cnt = sum(1 for s in all_results.sites if s.cas_type == 'SaCas9')
    cas12a_cnt = sum(1 for s in all_results.sites if s.cas_type == 'Cas12a')

    # SpCas9 9개 변종 개별 출력
    print(f"| SpCas9(WT)    |{sp_cnt:^15}|")
    print(f"| SpCas9-NG     |{sp_cnt:^15}|")
    print(f"| VRQR          |{sp_cnt:^15}|")
    print(f"| xCas          |{sp_cnt:^15}|")
    print(f"| Sniper        |{sp_cnt:^15}|")
    print(f"| SpCas9-HF.1   |{sp_cnt:^15}|")
    print(f"| eSpCas9(1.1)  |{sp_cnt:^15}|")
    print(f"| HypaCas9      |{sp_cnt:^15}|")
    print(f"| evoCas9       |{sp_cnt:^15}|")

    # 나머지 Cas 출력
    print(f"| SaCas9        |{sa_cnt:^15}|")
    print(f"| Cas12a        |{cas12a_cnt:^15}|")
    print("+---------------+---------------+")

    # 합계 대신 '총 예측 건수'로 명칭 변경 (1개 서열당 9번의 평가가 일어나므로)
    total_preds = (sp_cnt * 9) + sa_cnt + cas12a_cnt
    print(f"| 총 예측 건수  |{total_preds:^15}|")
    print("+---------------+---------------+")

    return all_results

import json
import numpy as np
import tensorflow as tf
from google.colab import drive
drive.mount('/content/drive')

base_path = '/content/drive/MyDrive/Colab Notebooks/project_CAS'
output_path = '/content/scanner_output.json'
model_path = '/content/model.keras'

# 모델 로드 (한 번만 로드)
model = tf.keras.models.load_model(model_path)
print("✅ 모델 로드 완료!")

targets = [
    'SpCas9', 'SpCas9-NG', 'VRQR variant', 'xCas',
    'Sniper-Cas9', 'SpCas9-HF.1', 'eSpCas9(1.1)', 'HypaCas9', 'evoCas9'
]

def one_hot_encode_dna(sequence):
    mapping = {'A': [1,0,0,0], 'C': [0,1,0,0], 'G': [0,0,1,0], 'T': [0,0,0,1]}
    return np.array([mapping.get(b, [0,0,0,0]) for b in sequence.upper()])



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 모델 로드 완료!


# 실행 코드

In [ ]:
from predict import load_models
import numpy as np
import torch
import json

model, models_sa, model_cas12a = load_models()

if __name__ == "__main__":
    output_path = '/content/scanner_output.json'

    # SpCas9 변종 9가지 이름 리스트 (모델 출력 인덱스 순서와 완벽 일치)
    SPCAS9_VARIANTS = [
        'SpCas9(WT)', 'SpCas9-NG', 'VRQR', 'xCas',
        'Sniper', 'SpCas9-HF.1', 'eSpCas9(1.1)', 'HypaCas9', 'evoCas9'
    ]

    while True:
        user_input = input("\n81bp DNA 서열을 입력하세요 (종료: q): ").strip()
        if user_input.lower() == 'q':
            print("프로그램 종료")
            break
        try:
            # 1. 스캐너 실행
            all_results = print_pam_analysis(user_input)

            # 2. 스캐너 결과 저장 (기존 기능 유지)
            scanner_results = []
            for site in all_results.sites:
                scanner_results.append({
                    'cas_type':       site.cas_type,
                    'guide_seq':      site.guide_seq,
                    'pam_seq':        site.pam_seq,
                    'model_input_seq': site.model_input_seq,
                    'distance':       site.distance,
                })

            with open(output_path, 'w') as f:
                json.dump(scanner_results, f)

            # 3. 모델 예측 + 가우시안 패널티 적용
            print("\n🤖 모델 예측 중...")

            # [수정] 9개 변종을 개별적으로 담기 위한 통합 랭킹 리스트
            final_display_results = []

            for site in all_results.sites:
                if not site.model_input_seq:
                    continue

                X = np.expand_dims(one_hot_encode_dna(site.model_input_seq), axis=0)
                penalty = gaussian_penalty(site.distance)

                if site.cas_type == 'SpCas9':

                    # 1. 모델이 뱉어내는 9개의 독립된 결과 리스트를 온전히 모두 받습니다.
                    preds = model.predict(X, verbose=0)

                    for i, variant_name in enumerate(SPCAS9_VARIANTS):
                        # 2. i번째 변종의 예측 결과 리스트에서 실제 소수점 값을 꺼냅니다.
                        # (preds[i]는 [[점수]] 형태의 2D 배열이므로 [0][0]으로 값을 추출합니다)
                        raw = float(preds[i][0][0])

                        final_score = raw * penalty * 100

                        final_display_results.append({
                            'cas_name': variant_name,
                            'pam_seq': site.pam_seq,
                            'guide_seq': site.guide_seq,
                            'raw': raw,
                            'distance': site.distance,
                            'final_score': final_score
                        })

                elif site.cas_type == 'SaCas9':
                    # 파이토치 5-Fold 앙상블
                    X_tensor = torch.tensor(X, dtype=torch.float32)
                    scores = []
                    with torch.no_grad():
                        for m in models_sa:
                            scores.append(float(m(X_tensor).item()))
                    raw = sum(scores) / len(scores)
                    final_score = raw * penalty * 100

                    final_display_results.append({
                        'cas_name': 'SaCas9',
                        'pam_seq': site.pam_seq,
                        'guide_seq': site.guide_seq,
                        'raw': raw,
                        'distance': site.distance,
                        'final_score': final_score
                    })

                elif site.cas_type == 'Cas12a':
                    raw = float(model_cas12a.predict(X, verbose=0)[0][0])
                    final_score = raw * penalty * 1  # Cas12a는 스케일 1 곱하기

                    final_display_results.append({
                        'cas_name': 'Cas12a',
                        'pam_seq': site.pam_seq,
                        'guide_seq': site.guide_seq,
                        'raw': raw,
                        'distance': site.distance,
                        'final_score': final_score
                    })

            # [수정] 통합된 모든 모델의 결과를 최종 점수(final_score) 기준으로 내림차순 정렬
            final_display_results.sort(key=lambda x: x['final_score'], reverse=True)

            # 4. 결과 출력
            print("\n📊 최종 예측 결과 (가우시안 패널티 적용)")
            for res in final_display_results:
                bar = '█' * int(res['final_score'] // 5)
                print(f"\n📍 Cas: {res['cas_name']} | PAM: {res['pam_seq']} | Guide: {res['guide_seq']}")
                print(f"  raw: {res['raw']:.4f} | 거리: {res['distance']} | 최종: {res['final_score']:.2f}%  {bar}")

        except ValueError as e:
            print(f"❌ 오류: {e}")
            print("다시 입력하세요.")


81bp DNA 서열을 입력하세요 (종료: q): ACACACACACACACACACACACACACACACACACATGGACACACACGACACACACACACACACACACACACACACACACAC

🔍 DNA 서열 분석 시작 (길이: 81bp)
------------------------------------------------------------
[SpCas9] 탐색 결과:
  · 가닥: + | PAM: TGG | 위치: 35 | 가이드: CACACACACACACACACACA
  · 가닥: + | PAM: GGA | 위치: 36 | 가이드: ACACACACACACACACACAT
  · 가닥: + | PAM: CGA | 위치: 45 | 가이드: CACACACACATGGACACACA
  · 가닥: - | PAM: TGT | 위치: 29 | 가이드: TGTGTGTGTGTGTGTGTGTG
  · 가닥: - | PAM: TGT | 위치: 31 | 가이드: TGTGTGTGTGTGTGTGTGTG
  · 가닥: - | PAM: CGT | 위치: 34 | 가이드: GTGTGTGTGTGTGTGTGTGT
  · 가닥: - | PAM: TGT | 위치: 36 | 가이드: GTGTGTGTGTGTGTGTGTCG
  · 가닥: - | PAM: TGT | 위치: 38 | 가이드: GTGTGTGTGTGTGTGTCGTG
  · 가닥: - | PAM: TGT | 위치: 40 | 가이드: GTGTGTGTGTGTGTCGTGTG
  · 가닥: - | PAM: TGT | 위치: 46 | 가이드: GTGTGTGTCGTGTGTGTCCA
  · 가닥: - | PAM: TGT | 위치: 48 | 가이드: GTGTGTCGTGTGTGTCCATG
  · 가닥: - | PAM: TGT | 위치: 50 | 가이드: GTGTCGTGTGTGTCCATGTG
  · 가닥: - | PAM: TGT | 위치: 52 | 가이드: GTCGTGTGTGTCCATGTGTG
  · 가닥: - | PAM: TGT | 위치: 54 |